In [1]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import polars as pl

from config import (
    SAML_D_VALIDATION_START,
    SAML_D_TEST_START,
    SAML_D_TEST_END_EXCLUSIVE
)

from src.data.loader import SAMLDDataLoader
from src.graph.temporal_split import SAMLDTemporalSplitter

In [2]:
### load standardized transactions
loader = SAMLDDataLoader()

# lazily scan transactions from interim Parquet dataset
transactions = loader.scan_interim()

# create temporal splitter instance
temporal_splitter = SAMLDTemporalSplitter(
    transactions = transactions,
    validation_start = SAML_D_VALIDATION_START,
    test_start = SAML_D_TEST_START,
    test_end_exclusive = SAML_D_TEST_END_EXCLUSIVE
)

In [3]:
### inspect transaction splits
split_summary = temporal_splitter.get_split_summary()

display(split_summary)

temporal_split,transaction_count,laundering_transaction_count,unique_sender_count,unique_receiver_count,minimum_timestamp,maximum_timestamp,laundering_rate
str,u64,u64,u64,u64,datetime[μs],datetime[μs],f64
"""history""",7048788,7036,236894,578208,2022-10-07 10:35:19,2023-05-31 23:59:59,0.000998
"""validation""",897243,1024,81220,254654,2023-06-01 00:00:06,2023-06-30 23:59:59,0.001141
"""test""",1550421,1806,104329,333545,2023-07-01 00:00:01,2023-08-22 23:59:54,0.001165
"""excluded""",8400,7,4504,4887,2023-08-23 00:00:01,2023-08-23 10:57:12,0.000833


In [4]:
### check if each transaction received a split
# full transaction count
full_transaction_count = (
    transactions
    .select(
        pl.len()
        .alias(
            'transaction_count'
        )
    )
    .collect(
        engine = 'streaming'
    )
    .item()
)

# sum of split transaction counts
split_transaction_counts = int(
    split_summary['transaction_count']
    .sum()
)

assert(
    split_transaction_counts == full_transaction_count
)

In [5]:
### validation and test account-discovery targets
discovery_summary = temporal_splitter.get_discovery_summary()

display(discovery_summary)

evaluation_window,window_start,window_end_exclusive,known_suspicious_account_count,candidate_account_count,future_suspicious_account_count,previously_known_future_account_count,new_future_suspicious_account_count,rankable_new_suspicious_account_count,unseen_new_suspicious_account_count,rankable_target_coverage,candidate_positive_rate
str,datetime[μs],datetime[μs],u64,u64,u64,u64,u64,u64,u64,f64,f64
"""validation""",2023-06-01 00:00:00,2023-07-01 00:00:00,5737,731188,958,166,792,678,114,0.856061,0.000927
"""test""",2023-07-01 00:00:00,2023-08-23 00:00:00,6529,773733,1660,293,1367,1232,135,0.901244,0.001592


In [6]:
### inspect validation targets
validation_targets = (
    temporal_splitter.build_future_suspicious_accounts(
        window_start = SAML_D_VALIDATION_START,
        window_end_exclusive = SAML_D_TEST_START
    )
    .collect(
        engine = 'streaming'
    )
)

display(
    validation_targets
    .sort(
        'first_future_suspicious_timestamp'
    )
    .head(10)
)

account,first_future_suspicious_timestamp,last_future_suspicious_timestamp,laundering_sender_event_count,laundering_receiver_event_count,laundering_account_event_count,known_before_window,seen_before_window,is_new_suspicious_account,is_rankable_new_suspicious_account
str,datetime[μs],datetime[μs],u64,u64,u64,bool,bool,bool,bool
"""4422964366""",2023-06-01 02:44:14,2023-06-02 11:31:47,1,1,2,false,true,true,true
"""8226556265""",2023-06-01 02:44:14,2023-06-14 22:27:35,1,1,2,false,true,true,true
"""6248605962""",2023-06-01 04:59:17,2023-06-06 11:33:49,0,8,8,true,true,false,false
"""8425259229""",2023-06-01 04:59:17,2023-06-01 04:59:17,1,0,1,false,true,true,true
"""6168003901""",2023-06-01 08:53:28,2023-06-08 07:22:33,3,0,3,true,true,false,false
"""2320891132""",2023-06-01 08:53:28,2023-06-07 14:42:25,0,2,2,false,true,true,true
"""5469723863""",2023-06-01 09:35:16,2023-06-01 09:35:16,1,0,1,true,true,false,false
"""3540377839""",2023-06-01 09:35:16,2023-06-01 09:35:16,0,1,1,false,false,true,false
"""5139808800""",2023-06-01 09:41:21,2023-06-07 14:42:25,4,0,4,true,true,false,false


In [7]:
### check target flag null summary
target_flag_null_summary = (
    validation_targets
    .select(
        [
            # known before window null count
            pl.col('known_before_window')
            .null_count()
            .alias(
                'known_before_window_null_count'
            ),
            # seen before window null count
            pl.col('seen_before_window')
            .null_count()
            .alias(
                'seen_before_window_null_count'
            ),
            # new suspicious account null count
            pl.col('is_new_suspicious_account')
            .null_count()
            .alias(
                'is_new_suspicious_account_null_count'
            ),
            # rankable target null count
            pl.col('is_rankable_new_suspicious_account')
            .null_count()
            .alias(
                'rankable_target_null_count'
            )
        ]
    )
)

display(target_flag_null_summary)

assert(
    target_flag_null_summary.row(0) == (0, 0, 0, 0)
)


known_before_window_null_count,seen_before_window_null_count,is_new_suspicious_account_null_count,rankable_target_null_count
u32,u32,u32,u32
0,0,0,0


In [8]:
### validation target flag summary
validation_target_flag_summary = (
    validation_targets
    .group_by(
        [
            'known_before_window',
            'seen_before_window',
            'is_new_suspicious_account',
            'is_rankable_new_suspicious_account'
        ]
    )
    .len(
        name = 'account_count'
    )
    .sort(
        [
            'known_before_window',
            'seen_before_window'
        ]
    )
)

display(validation_target_flag_summary)

known_before_window,seen_before_window,is_new_suspicious_account,is_rankable_new_suspicious_account,account_count
bool,bool,bool,bool,u32
false,false,true,false,114
false,true,true,true,678
true,true,false,false,166


In [9]:
### inspect only rankable validation targets
rankable_validation_targets = (
    validation_targets
    .filter(
        pl.col('is_rankable_new_suspicious_account')
    )
)

display(
    rankable_validation_targets
    .head(10)
)

account,first_future_suspicious_timestamp,last_future_suspicious_timestamp,laundering_sender_event_count,laundering_receiver_event_count,laundering_account_event_count,known_before_window,seen_before_window,is_new_suspicious_account,is_rankable_new_suspicious_account
str,datetime[μs],datetime[μs],u64,u64,u64,bool,bool,bool,bool
"""9810335545""",2023-06-20 18:28:36,2023-06-27 11:32:31,6,0,6,false,true,true,true
"""6864069292""",2023-06-10 13:07:03,2023-06-10 13:07:03,1,0,1,false,true,true,true
"""8623871346""",2023-06-02 20:55:42,2023-06-02 20:55:42,0,1,1,false,true,true,true
"""6019917057""",2023-06-14 16:58:18,2023-06-19 15:47:07,8,0,8,false,true,true,true
"""849759292""",2023-06-13 02:50:54,2023-06-13 02:50:54,1,0,1,false,true,true,true
"""2744104791""",2023-06-15 19:19:04,2023-06-15 19:19:04,0,1,1,false,true,true,true
"""9205869061""",2023-06-04 00:28:58,2023-06-05 13:26:16,1,1,2,false,true,true,true
"""9029593809""",2023-06-21 19:01:39,2023-06-21 19:01:39,1,0,1,false,true,true,true
"""9002058306""",2023-06-03 19:14:07,2023-06-03 19:14:07,0,1,1,false,true,true,true


In [10]:
### check invalid rankable targets
invalid_rankable_target_count = (
    validation_targets
    .filter(
        pl.col('known_before_window') & pl.col('is_rankable_new_suspicious_account')
    )
    .height
)

assert(
    invalid_rankable_target_count == 0
)

**Validation Summary**

Validation summary identifies 3 mutually exclusive account groups during the validation period.

Interpretation of flags:
- **known_before_window**:

        if True, the account is already historically suspicious and can be used as a PageRank seed
        o/w, no laundering activity involving that account was observed before the validation cutoff

- **seen_before_window**:

        if True, the account already exists as a node in the historical graph
        o/w, the account first appears during the validation period

- **is_new_suspicious_account**:

        if True, the account must satisfy:
            * it was not previously known as suspicious
            * it already existed in the historical graph
        o/w False

Account Groups:

1) New but unseen
    - 114 accounts
    - the graph cannot rank them, since they are not represented in the historical graph
2) New and rankable
    - 678 accounts
    - PageRank will attempt to rank them highly among all eligible historical accounts
    - Seed reachability and graph components should be checked during graph EDA
3) Previously known and recurring
    - 166 accounts
    - not new discoveries, but remain as a potential PageRank seed
    - while evaluating PageRank, these accounts should be removed from the final candidate ranking

Discovery Summary:

- all future suspicious accounts
    - future suspicious = previously known + new suspicious 
        
        => (958 = 166 + 792)

- all new suspicious accounts
    - new suspicious = rankable new + unseen new 
    
        => (792 = 678 + 114)

- rankable target coverage
    - coverage = rankable new / new suspicious
        
        => (0.856061 = 678 / 792)